# Annotation Validation — Coverage and Agreement against MEVA

This notebook validates the pipeline's event annotations against the MEVA ground truth in two directions, with
one setup, one judge and one set of credentials. It merges the earlier coverage notebook with the QA
`annotations-report` notebook, so the two checks are always run on the same data with the same judge.

| Part | Question | Unit | Direction | Output table |
|---|---|---|---|---|
| **A. Coverage** | Do the generated annotations describe the activities MEVA recorded? | one **(video, MEVA activity type)** pair | recall | `bronze.meva_event_coverage` |
| **B. Agreement** | Does each generated annotation agree with the MEVA activities happening in its time window? | one **generated event** | precision | `bronze.annotations_report` |

**Part A** ignores timing: an annotation covers an activity type if it describes it anywhere in the same video, and
one annotation is enough however many instances MEVA has. **Part B** keeps timing: each generated event is matched
to the MEVA activities whose frame ranges overlap its window (the matching stage that used to live outside this
repository and wrote `bronze.matched_pairs` is rebuilt here from `bronze.ground_truth`), and the judge scores how
well the annotation text agrees with those activities. A low coverage means the pipeline misses things; a low
agreement means it describes things that are not there.

**Workflow:**
1. Connect to the PostgreSQL `bronze` schema (env vars locally, Kaggle Secrets on Kaggle) and load one judge, local
   MLX on Apple Silicon or Groq elsewhere.
2. Part A: build the comparison set, judge every (video, activity type) pair, report coverage, write the table.
3. Part B: match every generated event to its overlapping MEVA activities, judge agreement with a reference-based
   rubric, report the match rate, write the table.
4. Put the two numbers side by side, overall and per video.

**Scope:** semantic checks only. Temporal IoU and spatial IoU are separate evaluation stages.


## 1. Environment and database connection

Install the libraries, then resolve credentials. The same code runs locally and on Kaggle: each secret is read
from an environment variable first (`.env`, loaded with python-dotenv) and from Kaggle Secrets as a fallback.

| Value | Env var (local `.env`) | Kaggle Secret |
|---|---|---|
| Database | `SUPABASE_DB_URL` — or `SUPABASE_DB_PASSWORD` (+ optional `SUPABASE_DB_USER` / `_HOST` / `_PORT` / `_NAME`) | `DB_PASSWORD` |
| LLM judge (Groq backend only) | `GROQ_API_KEY` | `groq` |
| Judge model (optional) | `GROQ_MODEL`, `MLX_JUDGE_MODEL` | — |
| Judge backend (optional) | `JUDGE_BACKEND` = `auto` / `mlx` / `groq` | — |

On macOS the judge runs locally and **no Groq key is needed**. See `.env.example` in the repository root.

In [1]:
import platform
import subprocess
import sys

# mlx-lm only exists on macOS; groq is installed everywhere so either backend can be forced.
packages = ["pandas", "sqlalchemy", "psycopg2-binary", "python-dotenv", "groq"]
if platform.system() == "Darwin":
    packages.append("mlx-lm")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Installed:", ", ".join(packages))

Installed: pandas, sqlalchemy, psycopg2-binary, python-dotenv, groq, mlx-lm


In [2]:
import json
import os
import platform
import re
from datetime import datetime, timezone

import pandas as pd
from sqlalchemy import URL, create_engine, text

try:  # local runs
    from dotenv import load_dotenv

    load_dotenv()
except ModuleNotFoundError:
    pass

try:  # Kaggle runs
    from kaggle_secrets import UserSecretsClient

    _kaggle_secrets = UserSecretsClient()
except Exception:
    _kaggle_secrets = None


def get_secret(env_var, kaggle_name=None, default=None, required=True):
    """Read a credential from the environment, then from Kaggle Secrets."""
    value = os.environ.get(env_var)

    if not value and _kaggle_secrets is not None and kaggle_name:
        try:
            value = _kaggle_secrets.get_secret(kaggle_name)
        except Exception:
            value = None

    if not value:
        value = default

    if not value and required:
        raise RuntimeError(
            f"Missing credential: set the {env_var} environment variable"
            + (f" or the Kaggle Secret '{kaggle_name}'." if kaggle_name else ".")
        )

    return value


print("Kaggle Secrets available:", _kaggle_secrets is not None)

Kaggle Secrets available: False


In [3]:
DATABASE_URL = os.environ.get("SUPABASE_DB_URL")

if not DATABASE_URL:
    DATABASE_URL = URL.create(
        drivername="postgresql+psycopg2",
        username=get_secret("SUPABASE_DB_USER", default="postgres.uqofqjychtjsvfioebod"),
        password=get_secret("SUPABASE_DB_PASSWORD", kaggle_name="DB_PASSWORD"),
        host=get_secret("SUPABASE_DB_HOST", default="aws-0-ap-south-1.pooler.supabase.com"),
        port=int(get_secret("SUPABASE_DB_PORT", default="5432")),
        database=get_secret("SUPABASE_DB_NAME", default="postgres"),
    )
elif DATABASE_URL.startswith("postgresql://"):
    DATABASE_URL = DATABASE_URL.replace("postgresql://", "postgresql+psycopg2://", 1)

engine = create_engine(DATABASE_URL, pool_pre_ping=True)
print("SQLAlchemy engine created.")

SQLAlchemy engine created.


In [4]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user;")).fetchone()

print("Connection successful.")
print("Database:", result[0])
print("User:", result[1])

Connection successful.
Database: postgres
User: postgres


## 2. Judge settings

The judge backend is chosen from the operating system: **Apple Silicon macOS runs the judge locally with MLX**
(`mlx-community/Qwen3-0.6B-8bit`, the same local-model approach notebook 05 uses for annotation), and any other
platform — Kaggle included — calls Groq. Set `JUDGE_BACKEND` to `mlx` or `groq` to override the detection.

The two backends are asked in different shapes. Groq judges a whole video in one call: the annotation list once,
then every MEVA activity type of that video. The local 0.6B model cannot answer about several activities at once
without copying its own output template, so it is asked one activity at a time.

In [ ]:
COVERAGE_THRESHOLD = 0.5   # judge score at which a MEVA activity counts as covered (0.5 = clearly implied)
COVERAGE_TABLE = "meva_event_coverage"
WRITE_TO_DB = True         # set to False to keep the report local
# None: judge every pair and rebuild the whole table. A substring of video_name (a full name for one
# video, a camera id such as "G421" for a camera): judge only those videos and replace only their rows,
# leaving the rest of the table as it is. Used to test a prompt change on a pilot video.
VIDEO_FILTER = None  # Pilot video. Set to None to judge everything.

# Part B (agreement) settings. The rubric is tagged with a prompt version so earlier versions written by the
# QA notebook (v1_explicit_judge, v2.3_semantic_judge) stay in the table for comparison; bump it when the rubric
# changes. An event matches when its score reaches AGREEMENT_THRESHOLD.
AGREEMENT_TABLE = "annotations_report"
MATCHED_PAIRS_TABLE = "matched_pairs"   # the event-to-MEVA matches behind Part B, kept for the QA notebook
AGREEMENT_PROMPT_VERSION = "v3_merged_semantic_judge"
AGREEMENT_THRESHOLD = 0.75
AGREEMENT_BATCH_SIZE = 20   # events per hosted call; the local judge always takes one event per call
FRAME_TOLERANCE = 0         # extra frames either side of an event window when matching MEVA activities

GROQ_JUDGE_MODEL = get_secret("GROQ_MODEL", default="openai/gpt-oss-120b", required=False)
# Set MLX_JUDGE_MODEL to override. Smaller local models were tried first and were not usable:
# Qwen3-0.6B-8bit marked every activity in a video as covered (see the warning it still prints).
MLX_JUDGE_MODEL = get_secret(
    "MLX_JUDGE_MODEL", default="mlx-community/Qwen3-8B-4bit", required=False
)


def resolve_judge_backend():
    """Local MLX on Apple Silicon macOS, Groq anywhere else. JUDGE_BACKEND overrides."""
    requested = os.environ.get("JUDGE_BACKEND", "auto").strip().lower()

    if requested in {"mlx", "groq"}:
        return requested

    if requested not in {"", "auto"}:
        raise ValueError(f"JUDGE_BACKEND must be auto, mlx or groq - got {requested!r}")

    is_apple_silicon = platform.system() == "Darwin" and platform.machine() == "arm64"
    return "mlx" if is_apple_silicon else "groq"


JUDGE_BACKEND = resolve_judge_backend()
JUDGE_MODEL = MLX_JUDGE_MODEL if JUDGE_BACKEND == "mlx" else GROQ_JUDGE_MODEL

print("Operating system: ", platform.system(), platform.machine())
print("Judge backend:    ", JUDGE_BACKEND, "(local)" if JUDGE_BACKEND == "mlx" else "(hosted)")
print("Judge model:      ", JUDGE_MODEL)
print("Asked per call:   ", "one activity" if JUDGE_BACKEND == "mlx" else "one whole video")
print("Coverage cutoff:  ", COVERAGE_THRESHOLD)
print("Coverage table:   ", f"bronze.{COVERAGE_TABLE}")
print("Agreement table:  ", f"bronze.{AGREEMENT_TABLE} (prompt {AGREEMENT_PROMPT_VERSION}, match at >= {AGREEMENT_THRESHOLD})")
print("Video filter:     ", VIDEO_FILTER or "none (all videos)")

if JUDGE_BACKEND == "mlx" and re.search(r"-(0\.6B|1\.7B|4B)-", JUDGE_MODEL):
    print(
        "\nWARNING: this local model is smaller than the tested default. Qwen3-0.6B-8bit, measured\n"
        "on camera G340's 13:15 video (15 MEVA activity types, 11 annotations), marked all 15 as\n"
        "covered and credited one annotation - 'White SUV enters from the left' - for most of them,\n"
        "scoring almost everything 0.75 with a canned reason. Splitting it into one question per\n"
        "annotation did not help. Check section 10.2 before reporting anything from a small judge."
    )

elif JUDGE_BACKEND == "mlx":
    print(
        "\nNote: a local judge is still weaker than the hosted 120B model. Check section 10.2\n"
        "before reporting these numbers, and re-run with JUDGE_BACKEND=groq to compare."
    )

Operating system:  Darwin arm64
Judge backend:     mlx (local)
Judge model:       mlx-community/Qwen3-8B-4bit
Asked per call:    one activity
Coverage cutoff:   0.5
Coverage table:    bronze.meva_event_coverage
Agreement table:   bronze.annotations_report (prompt v3_merged_semantic_judge, match at >= 0.75)
Video filter:      2018-03-07.11-00-01.11-05-01.school.G421

Note: a local judge is still weaker than the hosted 120B model. Check section 10.2
before reporting these numbers, and re-run with JUDGE_BACKEND=groq to compare.


# Part A — Coverage: do the annotations describe what MEVA recorded?

## 3. Build the comparison set

One row per (processed video, MEVA activity type). Only videos present in `bronze.videos` take part: MEVA has
ground truth for hundreds of videos the pipeline never ingested, and those are an ingestion gap rather than a
description failure, so they are left out entirely. `bronze.ground_truth` is collapsed to the distinct activity types of
each video — the instance count is kept as `n_meva_instances`, and the ground-truth ids as `gt_ids`, but they
play no part in the comparison. Every generated event of that video becomes a candidate annotation. No frame
condition is used anywhere.

Rows are tagged with a `status`:

* `judged` — the video has generated annotations, so the pair goes to the judge.
* `no_annotations` — the pipeline produced nothing for this video. Not covered by definition; no LLM call needed.


In [ ]:
# bronze.events gains `actions` (checklist phrases the annotator named) from notebook 07 onwards;
# older loads do not have the column, so it is only selected when present.
EVENTS_HAVE_ACTIONS = bool(
    pd.read_sql(
        text("""
        SELECT EXISTS (
            SELECT 1 FROM information_schema.columns
            WHERE table_schema = 'bronze' AND table_name = 'events' AND column_name = 'actions'
        ) AS has_actions;
        """),
        engine,
    ).iloc[0, 0]
)
ACTIONS_SQL = ",\n                'actions', COALESCE(e.actions, '[]'::jsonb)" if EVENTS_HAVE_ACTIONS else ""
print("bronze.events has an actions column:", EVENTS_HAVE_ACTIONS)

comparison_sql = text(f"""
WITH meva AS (
    SELECT
        g.video_id,
        g.camera_id,
        g.capture_start_local,
        g.activity_type,
        COUNT(*) AS n_meva_instances,
        array_agg(g.id ORDER BY g.id) AS gt_ids,
        MAX(v.video_name) AS video_name
    FROM bronze.ground_truth g
    JOIN bronze.videos v              -- only videos the pipeline processed and loaded
        ON v.video_id = g.video_id
    GROUP BY g.video_id, g.camera_id, g.capture_start_local, g.activity_type
),
annotations AS (
    SELECT
        e.video_id,
        COUNT(*) AS n_annotations,
        json_agg(
            json_build_object(
                'event_id', e.event_id,
                'event_name', e.event_name,
                'description', e.description{ACTIONS_SQL}
            )
            ORDER BY e.start_frame
        ) AS annotations
    FROM bronze.events e
    GROUP BY e.video_id
)
SELECT
    meva.video_id || ' | ' || meva.activity_type AS pair_id,
    meva.camera_id,
    meva.capture_start_local,
    meva.video_id,
    meva.video_name,
    meva.activity_type,
    meva.n_meva_instances,
    meva.gt_ids,
    COALESCE(annotations.n_annotations, 0) AS n_annotations,
    COALESCE(annotations.annotations, '[]') AS annotations
FROM meva
LEFT JOIN annotations ON annotations.video_id = meva.video_id
ORDER BY meva.camera_id, meva.capture_start_local, meva.activity_type;
""")

comparison_set = pd.read_sql(comparison_sql, engine)

comparison_set["annotations"] = comparison_set["annotations"].apply(
    lambda value: value if isinstance(value, list) else json.loads(value)
)

comparison_set["status"] = "judged"
comparison_set.loc[comparison_set["n_annotations"] == 0, "status"] = "no_annotations"

print("Processed videos with MEVA ground truth:", comparison_set["video_id"].nunique())
print("MEVA activity types to compare:", len(comparison_set))
print("MEVA instances behind them:   ", int(comparison_set["n_meva_instances"].sum()))
print()
print(comparison_set["status"].value_counts().to_string())

comparison_set[["video_name", "camera_id", "capture_start_local", "activity_type",
                "n_meva_instances", "n_annotations", "status"]].head(12)

### 3.1 Which objects each MEVA activity implies

MEVA ground truth stores an activity type and an actor count, not object labels, so the expected objects are
derived from the activity name itself: `person_opens_trunk` implies a person and a vehicle. The check then looks
for those object words in the generated annotation text.

The check runs **per annotation**, not over the video's pooled text. Pooling was useless: every video that has
annotations mentions both a person and some vehicle word somewhere, so nothing was ever reported missing.
Per annotation it is informative. `has_annotation_with_all_objects` is the yes/no answer — does at least one
of the video's annotations name every object the activity implies — and `n_annotations_with_all_objects` keeps
the count behind it. After judging, `covering_objects_missing` says whether the one annotation credited with
covering the activity actually named them.

In [7]:
OBJECT_KEYWORDS = {
    "person": ("person", "people", "man", "woman", "pedestrian", "individual", "someone"),
    "vehicle": ("vehicle", "car", "suv", "truck", "van", "sedan", "bus", "motorcycle", "hatchback"),
}


def meva_required_objects(activity_type):
    """Actor classes implied by a MEVA activity name."""
    activity = activity_type.lower()
    required = []

    if "person" in activity or "hand" in activity:
        required.append("person")
    if "vehicle" in activity or "trunk" in activity:
        required.append("vehicle")

    return required


def objects_missing_from_annotation(required, annotation):
    """Required object classes that this one annotation does not mention by name."""
    text = f"{annotation.get('event_name', '')} {annotation.get('description', '')}".lower()

    return [
        obj for obj in required
        if not any(keyword in text for keyword in OBJECT_KEYWORDS[obj])
    ]


def count_annotations_with_all_objects(required, annotations):
    """How many of the video's annotations name every object the activity implies."""
    return sum(
        1 for annotation in annotations
        if not objects_missing_from_annotation(required, annotation)
    )


comparison_set["required_objects"] = comparison_set["activity_type"].apply(meva_required_objects)
comparison_set["n_annotations_with_all_objects"] = comparison_set.apply(
    lambda row: count_annotations_with_all_objects(row["required_objects"], row["annotations"]),
    axis=1,
)
comparison_set["has_annotation_with_all_objects"] = comparison_set["n_annotations_with_all_objects"] > 0

comparison_set[["camera_id", "activity_type", "required_objects", "n_annotations",
                "has_annotation_with_all_objects", "n_annotations_with_all_objects",
                "status"]].head(12)

,camera_id,activity_type,required_objects,n_annotations,has_annotation_with_all_objects,n_annotations_with_all_objects,status
0,G299,person_talks_on_phone,[person],7,True,7,judged
1,G299,person_texts_on_phone,[person],7,True,7,judged
2,G299,person_picks_up_object,[person],0,False,0,no_video
3,G299,person_puts_down_object,[person],0,False,0,no_video
4,G299,person_sits_down,[person],0,False,0,no_video
5,G299,person_talks_to_person,[person],0,False,0,no_video
6,G299,person_carries_heavy_object,[person],0,False,0,no_video
7,G299,person_exits_scene_through_structure,[person],0,False,0,no_video
8,G299,person_picks_up_object,[person],0,False,0,no_video
9,G299,person_puts_down_object,[person],0,False,0,no_video


## 4. Pairs to evaluate

Every pair in the comparison set is judged on every run, and section 11 rebuilds `bronze.meva_event_coverage`
from the result. Nothing is skipped as "already evaluated": re-annotated videos, new ground truth and a switch
of judge model are all picked up simply by running the notebook again. The cost is one full pass of judge
calls per run.

With `VIDEO_FILTER` set (section 2) only the matching videos are judged, and section 11 replaces just their
rows, so a prompt change can be tested on one pilot video without re-judging or wiping the others.

The earlier frame-based design wrote `bronze.meva_coverage_report`, which has a different shape (one row per
ground-truth instance, with frame columns and a frame tolerance); that table is left as it is.


In [8]:
report_exists = pd.read_sql(
    text("""
    SELECT EXISTS (
        SELECT 1
        FROM information_schema.tables
        WHERE table_schema = 'bronze'
          AND table_name = :table_name
    ) AS report_exists;
    """),
    engine,
    params={"table_name": COVERAGE_TABLE},
).iloc[0, 0]

existing_rows = 0
if report_exists:
    if VIDEO_FILTER:
        existing_rows = int(pd.read_sql(
            text(f"SELECT COUNT(*) FROM bronze.{COVERAGE_TABLE} WHERE video_name LIKE :pattern;"),
            engine, params={"pattern": f"%{VIDEO_FILTER}%"},
        ).iloc[0, 0])
    else:
        existing_rows = int(
            pd.read_sql(text(f"SELECT COUNT(*) FROM bronze.{COVERAGE_TABLE};"), engine).iloc[0, 0]
        )

# Every selected pair is evaluated on every run; section 11 replaces its rows.
if VIDEO_FILTER:
    pending = comparison_set[
        comparison_set["video_name"].str.contains(VIDEO_FILTER, regex=False, na=False)
    ].copy()
    if pending.empty:
        raise ValueError(f"No MEVA pairs match VIDEO_FILTER={VIDEO_FILTER!r}; check the name in bronze.videos.")
    print(f"VIDEO_FILTER={VIDEO_FILTER!r}: {pending['video_name'].nunique()} video(s) selected.")
else:
    pending = comparison_set.copy()

pending_judged = pending[pending["status"] == "judged"].copy()
pending_auto = pending[pending["status"] != "judged"].copy()

print("Report table exists:      ", bool(report_exists))
print("Rows to be replaced:      ", existing_rows if WRITE_TO_DB else f"{existing_rows} (kept, WRITE_TO_DB is False)")
print("Pending evaluation:       ", len(pending))
print("  -> sent to the judge:   ", len(pending_judged))
print("  -> resolved without LLM:", len(pending_auto))


VIDEO_FILTER='2018-03-07.11-00-01.11-05-01.school.G421': 1 video(s) selected.
Report table exists:       True
Rows to be replaced:       14
Pending evaluation:        14
  -> sent to the judge:    14
  -> resolved without LLM: 0


## 5. Load the judge

Only the selected backend is initialised, so a macOS run never needs a Groq key and a Kaggle run never needs MLX.
The local model is downloaded from Hugging Face on first use and cached afterwards.

In [9]:
groq_client = None
mlx_model = None
mlx_tokenizer = None
mlx_sampler = None
MLX_MAX_TOKENS = 300

if JUDGE_BACKEND == "groq":
    from groq import Groq

    groq_client = Groq(api_key=get_secret("GROQ_API_KEY", kaggle_name="groq"))

    response = groq_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": "Reply exactly: judge connection working"}],
        temperature=0,
    )
    print("Groq:", response.choices[0].message.content)

else:
    from mlx_lm import generate as mlx_generate
    from mlx_lm import load as mlx_load
    from mlx_lm.sample_utils import make_sampler

    print(f"Loading local judge: {JUDGE_MODEL}", flush=True)
    mlx_model, mlx_tokenizer = mlx_load(JUDGE_MODEL)
    mlx_sampler = make_sampler(temp=0.0)
    print("Local judge ready.")

/Users/juanvargas/UTS/3rd Semester/Capstone/annotation_pipeline/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading local judge: mlx-community/Qwen3-8B-4bit


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 2293.64it/s]


Local judge ready.


## 6. The judge

Both backends get the same rules; only the shape differs.

| Score | Meaning |
|---|---|
| 1.00 | An annotation explicitly describes this activity, with the same kind of actors |
| 0.75 | An annotation describes the same action in different wording, or a close variant of it |
| 0.50 | No annotation names the action, but what is described clearly implies it |
| 0.25 | The right actors or scene appear, but nothing suggests this action |
| 0.00 | No annotation relates to this activity |

An activity counts as **covered** from `COVERAGE_THRESHOLD` upwards (section 2, default `0.5`), so a clearly
implied action is enough. The rules deliberately leave room for paraphrase and reasonable inference: a
direction change counts as a turn without the word "turn", leaning into a trunk counts for `person_opens_trunk`,
and the vehicle type or head-count never has to match. What is still refused is coverage by name alone (the
activity name is a hint about what to look for, not evidence), generic movement standing in for a specific
manoeuvre, and proximity standing in for an interaction.

Annotations produced with `videometa>=0.0.17` carry an `actions` list, the checklist phrases the annotator named
(`opens a vehicle door`, `vehicle reverses`, ...). It is shown to the judge next to the description as part of
the annotation: a listed action scores 1.00 when the description backs it up and 0.75 when only the list names it.

Timing is explicitly declared irrelevant in the prompt — the activity may be described anywhere in the video.


In [10]:
JUDGE_RULES = """coverage_score scale:
1.00 - an annotation explicitly describes this activity, with the same kind of actors
0.75 - an annotation describes the same action in different words, or a close variant of it
       ("gets out of the car" for person_exits_vehicle, "pulls out of the parking spot" for vehicle_starts)
0.50 - no annotation names the action, but what is described clearly implies it: the right actors in the
       same situation, where the action is the natural reading ("walks up to the SUV and the door swings
       open" for person_opens_vehicle_door, "slows down and pulls over" for vehicle_stops)
0.25 - the right actors or scene appear, but nothing suggests this action
0.00 - no annotation relates to this activity

Rules:
- Timing does not matter. The annotations are unordered and may describe the activity at any point in the video.
- How many annotations there are does not matter. One annotation describing the activity is enough.
- Wording does not matter. Credit synonyms, paraphrases and everyday phrasing of the same action.
- Reasonable inference is allowed. Credit an action the description clearly implies (0.50), but not one that
  merely could have happened off-screen or that only the activity name suggests.
- Generic movement alone ("moves", "drives past", "enters the frame") does not describe a specific manoeuvre.
  But a change of direction, slowing, pulling in or out, backing up or swinging around counts as turning,
  stopping, starting, reversing or a U-turn even without that exact word.
- Standing near something, on its own, does not describe opening, closing, entering, exiting, picking up,
  putting down or carrying. Interacting with it does: reaching into, leaning into, loading, climbing in or
  out, holding or handing over all count for the matching activity.
- The actors should match in kind: an activity about a vehicle needs a vehicle in the annotation, one about a
  person needs a person. The exact vehicle type, colour or number of people does not matter.
- Judge the annotation text, not the activity name: the name tells you what to look for, not that it happened.
- An annotation's actions list, when present, is part of the annotation text: a listed action is explicitly
  described (1.00) when the description also mentions it, and the same action in other words (0.75) when
  only the list names it."""


def numbered_annotations(annotations):
    lines = []
    for position, annotation in enumerate(annotations, start=1):
        line = f'{position}. event_name: "{annotation["event_name"]}" | description: "{annotation["description"]}"'
        actions = annotation.get("actions") or []
        if actions:
            line += f' | actions: {"; ".join(str(action) for action in actions)}'
        lines.append(line)
    return "\n".join(lines)


def build_groq_prompt(video_rows):
    """One call per video: the annotations once, then every MEVA activity type of that video."""
    first = video_rows.iloc[0]
    activity_types = list(video_rows["activity_type"])

    return f"""You are a fair evaluator checking whether the events an automatic annotation pipeline detected
in one CCTV video COVER the activities that human MEVA annotators recorded for that same video. Give credit
whenever the annotation text genuinely describes or clearly implies the activity, whatever the wording.

{JUDGE_RULES}

covered is true only when coverage_score >= {COVERAGE_THRESHOLD}.
covering_event_id is the event_id of the single best annotation when covered is true, otherwise null.

GENERATED ANNOTATIONS FOR VIDEO {first["camera_id"]} ({first["capture_start_local"]}):
{numbered_annotations(first["annotations"])}

MEVA ACTIVITY TYPES RECORDED IN THIS VIDEO:
{json.dumps(activity_types, indent=2)}

Judge every activity type in that list, in order. Return ONLY a valid JSON array, one object per activity type:

[
  {{
    "activity_type": "<the activity type, copied exactly>",
    "coverage_score": <one of 1.0, 0.75, 0.5, 0.25, 0.0>,
    "covered": <true or false>,
    "covering_event_id": <event_id or null>,
    "reason": "<one short sentence>"
  }}
]
"""


def build_mlx_prompt(row):
    """One call per activity type, numbered annotations, index-based answer."""
    return f"""You are a fair evaluator. You are given ONE activity type that human annotators recorded in a
CCTV video, and all the events an automatic pipeline detected in that same video.

Decide whether ANY of the numbered annotations describes or clearly implies that activity. Give credit
whenever the text genuinely does, whatever the wording.

{JUDGE_RULES}

Answer with one JSON object and nothing else:
  "coverage_score": one of 1.0, 0.75, 0.5, 0.25, 0.0
  "covered": true only when coverage_score is {COVERAGE_THRESHOLD} or higher
  "covering_index": the number of the annotation that describes the activity, or 0 if none does
  "reason": one short sentence explaining your score

MEVA ACTIVITY: {row["activity_type"]}

GENERATED ANNOTATIONS:
{numbered_annotations(row["annotations"])}

JSON object:"""

### 6.1 Normalising a verdict

Whatever the backend returns is reduced to the same five fields. The score is snapped to the nearest allowed
value, `covered` is derived from the score rather than taken from the model, and a `covering_event_id` is kept
only when it really is one of that video's annotations — a model asked for an event id will otherwise answer
with the MEVA activity name. A verdict with no usable score is discarded, leaving that pair pending.

In [11]:
ALLOWED_SCORES = (0.0, 0.25, 0.5, 0.75, 1.0)


def normalise_verdict(pair_id, score, reason, covering_event_id, annotations):
    """Return a clean verdict record, or None when the response is unusable."""
    try:
        score = min(ALLOWED_SCORES, key=lambda allowed: abs(allowed - float(score)))
    except (TypeError, ValueError):
        return None

    covered = score >= COVERAGE_THRESHOLD

    valid_event_ids = {annotation["event_id"] for annotation in annotations}
    if not covered or covering_event_id not in valid_event_ids:
        covering_event_id = None

    return {
        "pair_id": pair_id,
        "coverage_score": score,
        "covered": covered,
        "covering_event_id": covering_event_id,
        "reason": str(reason or "").strip(),
    }

## 7. Parse the judge output

Both backends are asked for JSON. The helpers strip Markdown fences and any `<think>` block (Qwen3 emits
reasoning tags even with thinking disabled) before extracting the array or object from the response.

In [12]:
def strip_wrappers(response_text):
    """Remove reasoning tags and Markdown fences."""
    cleaned = re.sub(r"<think>.*?</think>", "", response_text, flags=re.DOTALL)
    cleaned = cleaned.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    return cleaned.strip()


def parse_json_array(response_text):
    cleaned = strip_wrappers(response_text)

    if not cleaned.startswith("["):
        match = re.search(r"\[.*\]", cleaned, re.DOTALL)
        if not match:
            raise ValueError("No JSON array in judge response.")
        cleaned = match.group(0)

    parsed = json.loads(cleaned)

    if not isinstance(parsed, list):
        raise ValueError("Judge response is not a JSON array.")

    return parsed


def parse_json_object(response_text):
    match = re.search(r"\{.*?\}", strip_wrappers(response_text), re.DOTALL)

    if not match:
        raise ValueError("No JSON object in judge response.")

    return json.loads(match.group(0))

## 8. Judge the pending pairs

`judge_pending` dispatches to the selected backend: Groq one video per call, MLX one activity per call. Either
way each call is retried once when the response is not parsable, and anything still unusable is left pending
instead of being scored. Videos with no annotations never reach the judge.

In [13]:
def judge_groq_video(video_rows):
    """Judge every MEVA activity type of one video in a single hosted call."""
    response = groq_client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": build_groq_prompt(video_rows)}],
        temperature=0,
    )
    verdicts = parse_json_array(response.choices[0].message.content)

    # Map the judge's activity_type back to our pair, and ignore anything we did not ask about
    # (the output template carries a placeholder the model sometimes echoes).
    by_activity = {
        str(row["activity_type"]).strip().lower(): row
        for _, row in video_rows.iterrows()
    }
    annotations = video_rows.iloc[0]["annotations"]

    normalised = []
    for verdict in verdicts:
        row = by_activity.get(str(verdict.get("activity_type", "")).strip().lower())
        if row is None:
            continue

        clean = normalise_verdict(
            row["pair_id"],
            verdict.get("coverage_score"),
            verdict.get("reason"),
            verdict.get("covering_event_id"),
            annotations,
        )
        if clean:
            normalised.append(clean)

    return normalised


def judge_mlx_pair(row):
    """Judge one activity type with the local model."""
    output = mlx_generate(
        mlx_model,
        mlx_tokenizer,
        prompt=mlx_tokenizer.apply_chat_template(
            [{"role": "user", "content": build_mlx_prompt(row)}],
            add_generation_prompt=True,
            enable_thinking=False,
        ),
        max_tokens=MLX_MAX_TOKENS,
        sampler=mlx_sampler,
        verbose=False,
    )
    verdict = parse_json_object(output)

    # The local model answers with a position in the numbered list, not an event id.
    annotations = row["annotations"]
    index = verdict.get("covering_index")
    covering_event_id = None
    if isinstance(index, int) and 1 <= index <= len(annotations):
        covering_event_id = annotations[index - 1]["event_id"]

    clean = normalise_verdict(
        row["pair_id"],
        verdict.get("coverage_score"),
        verdict.get("reason"),
        covering_event_id,
        annotations,
    )
    if clean is None:
        raise ValueError(f"Unusable verdict: {verdict}")

    return [clean]


def judge_pending(pending_df):
    """Judge every pending pair with the selected backend, retrying each call once."""
    verdicts = []
    failures = []

    if JUDGE_BACKEND == "mlx":
        units = [row for _, row in pending_df.iterrows()]
        run = judge_mlx_pair
        describe = lambda unit: f"{unit['camera_id']} / {unit['activity_type']}"
    else:
        units = [
            group for _, group in pending_df.groupby(["camera_id", "capture_start_local"], sort=True)
        ]
        run = judge_groq_video
        describe = lambda unit: (
            f"{unit.iloc[0]['camera_id']} {unit.iloc[0]['capture_start_local']}"
            f" ({len(unit)} activity types)"
        )

    for position, unit in enumerate(units, start=1):
        label = f"{position}/{len(units)} {describe(unit)}"

        for attempt in (1, 2):
            try:
                verdicts.extend(run(unit))
                break
            except Exception as error:
                print(f"  {label} attempt {attempt} failed: {error}")
                if attempt == 2:
                    failures.append(label)

        if JUDGE_BACKEND == "groq" or position % 20 == 0 or position == len(units):
            print(f"Judged {label}", flush=True)

    return verdicts, failures


judgements, failed_calls = judge_pending(pending_judged)

print(f"\nPairs sent to the judge:   {len(pending_judged)}")
print(f"Verdicts returned:         {len(judgements)}")
print(f"Auto-resolved without LLM: {len(pending_auto)}")
if failed_calls:
    print("Failed calls:", failed_calls)

Judged 14/14 G421 / person_transfers_object

Pairs sent to the judge:   14
Verdicts returned:         14
Auto-resolved without LLM: 0


## 9. Build the coverage report

The verdicts are merged back onto the pending pairs by `pair_id`, and the pairs that never went to the judge are
filled in: `no_annotations` becomes score `0.0` / not covered. A judged pair that came back without a usable verdict becomes `judge_missing` and stays pending.

In [ ]:
judgements_df = pd.DataFrame(
    judgements,
    columns=["pair_id", "coverage_score", "covered", "covering_event_id", "reason"],
).drop_duplicates(subset="pair_id", keep="last")

coverage_report = pending.merge(judgements_df, on="pair_id", how="left")

no_annotations = coverage_report["status"] == "no_annotations"
coverage_report.loc[no_annotations, "coverage_score"] = 0.0
coverage_report.loc[no_annotations, "covered"] = False
coverage_report.loc[no_annotations, "reason"] = (
    "The pipeline generated no annotations for this video."
)

missing_verdict = (
    (coverage_report["status"] == "judged") & coverage_report["coverage_score"].isna()
)
coverage_report.loc[missing_verdict, "status"] = "judge_missing"

coverage_report["covered"] = coverage_report["covered"].astype("object")
coverage_report["judge_backend"] = JUDGE_BACKEND
coverage_report["judge_model"] = JUDGE_MODEL
coverage_report["evaluated_at"] = datetime.now(timezone.utc)

annotation_names = {
    annotation["event_id"]: annotation["event_name"]
    for annotations in coverage_report["annotations"]
    for annotation in annotations
}
coverage_report["covering_event_name"] = coverage_report["covering_event_id"].map(annotation_names)

# Object check against the one annotation credited with covering the activity.
annotations_by_id = {
    annotation["event_id"]: annotation
    for annotations in coverage_report["annotations"]
    for annotation in annotations
}


def covering_objects_missing(row):
    annotation = annotations_by_id.get(row["covering_event_id"])
    if annotation is None:
        return None  # nothing was credited, so there is nothing to check
    return objects_missing_from_annotation(row["required_objects"], annotation)


coverage_report["covering_objects_missing"] = coverage_report.apply(covering_objects_missing, axis=1)

coverage_report["gt_ids_text"] = coverage_report["gt_ids"].apply(
    lambda ids: ",".join(str(gt_id) for gt_id in (ids or []))
)
coverage_report["required_objects_text"] = coverage_report["required_objects"].apply(",".join)
coverage_report["covering_objects_missing_text"] = coverage_report["covering_objects_missing"].apply(
    lambda objs: None if objs is None else ",".join(objs)
)

print("Report rows:", len(coverage_report))
print(coverage_report["status"].value_counts().to_string())
print("Pairs without a verdict:", int(missing_verdict.sum()))

In [15]:
coverage_report[
    [
        "video_name",
        "camera_id",
        "activity_type",
        "n_meva_instances",
        "n_annotations",
        "coverage_score",
        "covered",
        "covering_event_name",
        "covering_objects_missing_text",
        "status",
        "reason",
    ]
].head(30)

,video_name,camera_id,activity_type,n_meva_instances,n_annotations,coverage_score,covered,covering_event_name,covering_objects_missing_text,status,reason
0,2018-03-07.11-00-01.11-05-01.school.G421,G421,person_carries_heavy_object,2,105,0.50,True,Person walks toward the exit,,judged,Annotation 3 describes a person carrying a bag...
1,2018-03-07.11-00-01.11-05-01.school.G421,G421,person_enters_scene_through_structure,6,105,1.00,True,Person walks toward a table,,judged,Annotation 9 describes a person entering throu...
2,2018-03-07.11-00-01.11-05-01.school.G421,G421,person_exits_scene_through_structure,4,105,0.50,True,Woman walks toward the counter,,judged,The annotation describes a person walking towa...
3,2018-03-07.11-00-01.11-05-01.school.G421,G421,person_opens_facility_door,7,105,0.50,True,Man enters through the doors and walks toward ...,,judged,The annotation describes a person entering thr...
4,2018-03-07.11-00-01.11-05-01.school.G421,G421,person_picks_up_object,62,105,0.50,True,Woman at a table pours a drink into a cup,,judged,The annotation describes a woman pouring a dri...
5,2018-03-07.11-00-01.11-05-01.school.G421,G421,person_purchases,4,105,0.50,True,Woman walks from the counter toward a table,,judged,The annotation describes a woman handing somet...
6,2018-03-07.11-00-01.11-05-01.school.G421,G421,person_puts_down_object,73,105,0.50,True,Person walks toward the exit,,judged,The annotation describes a person carrying an ...
7,2018-03-07.11-00-01.11-05-01.school.G421,G421,person_reads_document,7,105,0.00,False,NaN,NaN,judged,None of the annotations describe or imply that...
8,2018-03-07.11-00-01.11-05-01.school.G421,G421,person_sits_down,7,105,1.00,True,Woman walks to a table and sits down,,judged,Annotation 70 explicitly describes the activit...
9,2018-03-07.11-00-01.11-05-01.school.G421,G421,person_stands_up,4,105,0.50,True,Person stands and looks at a phone,,judged,The annotation describes a person standing and...


## 10. Coverage metrics

Two headline numbers:

* **Activity coverage** — of the (video, activity type) pairs MEVA recorded, how many the pipeline described.
  This is the event comparison itself: each kind of activity in each video counts once.
* **Instance-weighted coverage** — the same verdicts weighted by how many times MEVA annotated each activity, so
  a missed activity that happened nine times counts nine times. Useful as a second view; it is not a count match.

Numbers from the local MLX judge and from Groq are not interchangeable,
so every stored row carries its `judge_backend` and `judge_model` and the final summary groups by them.

In [ ]:
evaluable = coverage_report[coverage_report["status"].isin(["judged", "no_annotations"])].copy()
evaluable["covered"] = evaluable["covered"].fillna(False).astype(bool)

pairs = len(evaluable)
covered_pairs = int(evaluable["covered"].sum())
instances = int(evaluable["n_meva_instances"].sum())
covered_instances = int(evaluable.loc[evaluable["covered"], "n_meva_instances"].sum())

print(f"MEVA activity types compared: {pairs}")
print(f"Covered by an annotation:     {covered_pairs}")
print(f"Activity coverage:            {covered_pairs / pairs:.1%}" if pairs else "n/a")
print()
print(f"MEVA instances behind them:   {instances}")
print(f"Instance-weighted coverage:   {covered_instances / instances:.1%}" if instances else "n/a")
print()
print("Misses by kind:")
print(f"  video had no annotations at all:     {int((evaluable['status'] == 'no_annotations').sum())}")
print(f"  annotations describe something else: "
      f"{int(((evaluable['status'] == 'judged') & ~evaluable['covered']).sum())}")


In [17]:
by_activity = (
    evaluable.groupby("activity_type")
    .agg(videos=("pair_id", "count"), covered=("covered", "sum"),
         meva_instances=("n_meva_instances", "sum"))
    .assign(coverage=lambda df: (df["covered"] / df["videos"]).round(3))
    .sort_values(["coverage", "videos"], ascending=[True, False])
)

print("Coverage by MEVA activity type (worst first):")
by_activity

Coverage by MEVA activity type (worst first):


,videos,covered,meva_instances,coverage
activity_type,,,,
person_reads_document,1,0,7,0.0
person_texts_on_phone,1,0,37,0.0
person_carries_heavy_object,1,1,2,1.0
person_enters_scene_through_structure,1,1,6,1.0
person_exits_scene_through_structure,1,1,4,1.0
person_opens_facility_door,1,1,7,1.0
person_picks_up_object,1,1,62,1.0
person_purchases,1,1,4,1.0
person_puts_down_object,1,1,73,1.0


In [18]:
by_video = (
    evaluable.groupby(["camera_id", "capture_start_local"])
    .agg(activity_types=("pair_id", "count"), covered=("covered", "sum"),
         annotations=("n_annotations", "max"))
    .assign(coverage=lambda df: (df["covered"] / df["activity_types"]).round(3))
    .sort_values("coverage")
)

print("Coverage by video (worst first):")
by_video

Coverage by video (worst first):


,,activity_types,covered,annotations,coverage
camera_id,capture_start_local,,,,
G421,2018-03-07 11:00:01,14,12,105,0.857


In [19]:
missed = evaluable[~evaluable["covered"]][
    ["camera_id", "capture_start_local", "activity_type", "n_meva_instances",
     "n_annotations", "has_annotation_with_all_objects", "coverage_score", "reason"]
].sort_values(["camera_id", "activity_type"])

print(f"MEVA activity types the pipeline did not cover: {len(missed)}")
missed.head(40)

MEVA activity types the pipeline did not cover: 2


,camera_id,capture_start_local,activity_type,n_meva_instances,n_annotations,has_annotation_with_all_objects,coverage_score,reason
7,G421,2018-03-07 11:00:01,person_reads_document,7,105,True,0.00,None of the annotations describe or imply that...
12,G421,2018-03-07 11:00:01,person_texts_on_phone,37,105,True,0.25,None of the annotations describe or imply that...


### 10.1 Object check

For each MEVA event, did the generated annotations name every object the activity implies
(`person_opens_trunk` implies a person and a vehicle)? Two totals: events where the annotations
found all the implied objects, and events where they missed at least one.

In [20]:
found_all_objects = coverage_report["has_annotation_with_all_objects"]

print("Events where the annotations found all the objects:", int(found_all_objects.sum()))
print("Events where the annotations missed at least one:  ", int((~found_all_objects).sum()))

Events where the annotations found all the objects: 14
Events where the annotations missed at least one:   0


Same two totals restricted to the events the judge marked as covered, checking only the single annotation it
credited. This is the stricter view: it asks whether the annotation that supposedly describes the activity names
the right objects.

In [21]:
credited = coverage_report[coverage_report["covering_objects_missing"].notna()]
credited_missing = credited["covering_objects_missing"].apply(len) > 0

print("Credited annotations checked:              ", len(credited))
print("  found all the objects:                  ", int((~credited_missing).sum()))
print("  missed at least one:                    ", int(credited_missing.sum()))

Credited annotations checked:               12
  found all the objects:                   12
  missed at least one:                     0


In [22]:
if credited_missing.any():
    print("Events whose credited annotation missed an implied object:")
    display(
        credited[credited_missing][
            [
                "camera_id",
                "activity_type",
                "required_objects_text",
                "covering_objects_missing_text",
                "covering_event_name",
                "judge_reason" if "judge_reason" in credited.columns else "reason",
            ]
        ].sort_values(["camera_id", "activity_type"])
    )
else:
    print("No credited annotation missed an implied object.")

No credited annotation missed an implied object.


### 10.2 Judge sanity check

Two signatures of a judge that is rubber-stamping instead of reading, both seen for real with the local 0.6B
model:

* **One annotation credited with many different activities in the same video.** "A car appears in the
  middle-right" cannot be what describes `person_enters_vehicle`, `vehicle_reverses` *and*
  `person_enters_scene_through_structure`.
* **A flat score distribution**, especially almost everything at exactly 0.75, which is the model repeating the
  middle of the scale rather than judging.

These checks do not change any verdict; they tell you whether the coverage number above is worth reporting.

In [23]:
judged_only = evaluable[evaluable["status"] == "judged"]

print("Score distribution over judged pairs:")
print(judged_only["coverage_score"].value_counts().sort_index().to_string())

if len(judged_only):
    counts = judged_only["coverage_score"].value_counts()
    levels_used = len(counts)

    print(f"\nScore levels used: {levels_used} of {len(ALLOWED_SCORES)}")

    if levels_used <= 2:
        print(
            f"WARNING: the judge only ever answered {sorted(counts.index)}. Two levels can mean the"
            " pipeline really is uniformly good or bad, but with a small model it usually means the"
            " judge is picking a fixed answer rather than grading. Compare against another judge"
            " before reporting."
        )

reused = (
    evaluable[evaluable["covered"] & evaluable["covering_event_id"].notna()]
    .groupby(["camera_id", "capture_start_local", "covering_event_name"])
    .agg(activity_types=("activity_type", "nunique"),
         activities=("activity_type", lambda values: ", ".join(sorted(values))))
    .reset_index()
    .sort_values("activity_types", ascending=False)
)
reused = reused[reused["activity_types"] >= 3]

if len(reused):
    print(
        f"\nWARNING: {len(reused)} annotation(s) are credited with covering 3 or more different MEVA"
        " activities in the same video. Check these before reporting the coverage number:"
    )
    display(reused.head(10))
else:
    print("\nNo annotation is credited with covering 3 or more different activities in one video.")

Score distribution over judged pairs:
coverage_score
0.00     1
0.25     1
0.50    10
1.00     2

Score levels used: 4 of 5



,camera_id,capture_start_local,covering_event_name,activity_types,activities
3,G421,2018-03-07 11:00:01,Person walks toward the exit,3,"person_carries_heavy_object, person_puts_down_..."


## 11. Write to `bronze.meva_event_coverage`

Without `VIDEO_FILTER` the table is dropped and rebuilt from this run's report, so it only ever holds the newest
verdicts — old rows from earlier runs or other judge models are removed. With `VIDEO_FILTER` set, only the
matching videos' rows are deleted and rewritten, in one transaction, and every other row stays. Only pairs
carrying a verdict are written; `judge_missing` rows are left out and a re-run retries them. Set
`WRITE_TO_DB = False` in section 2 to keep the report local and leave the stored table untouched.


In [24]:
import csv
import io

from sqlalchemy import BigInteger, Boolean, DateTime, Float, Integer, Text


def psql_insert_copy(table, conn, keys, data_iter):
    """to_sql insert method that streams rows with COPY instead of a multi-row INSERT.

    One INSERT carrying every row (~5,600 rows, 120k+ bind parameters) hits Supabase's 2-minute
    statement timeout, and chunked INSERTs take ~18 s per 500 rows over the pooler. COPY writes the
    same rows in well under a minute. Empty CSV fields are read back as NULL.
    """
    buffer = io.StringIO()
    csv.writer(buffer).writerows(data_iter)
    buffer.seek(0)

    columns = ", ".join(f'"{key}"' for key in keys)
    name = f"{table.schema}.{table.name}" if table.schema else table.name
    with conn.connection.cursor() as cursor:
        cursor.copy_expert(f"COPY {name} ({columns}) FROM STDIN WITH CSV", buffer)


REPORT_COLUMNS = [
    "pair_id",
    "camera_id",
    "capture_start_local",
    "video_id",
    "video_name",
    "activity_type",
    "n_meva_instances",
    "gt_ids_text",
    "required_objects_text",
    "covering_objects_missing_text",
    "n_annotations",
    "has_annotation_with_all_objects",
    "n_annotations_with_all_objects",
    "coverage_score",
    "covered",
    "covering_event_id",
    "covering_event_name",
    "reason",
    "status",
    "judge_backend",
    "judge_model",
    "evaluated_at",
]

REPORT_RENAMES = {
    "gt_ids_text": "gt_ids",
    "required_objects_text": "required_objects",
    "covering_objects_missing_text": "covering_objects_missing",
    "reason": "judge_reason",
}

# Explicit types: `covered` carries NULLs for unevaluable pairs, and without this
# to_sql would create it as TEXT the first time the table is built.
REPORT_DTYPES = {
    "pair_id": Text(),
    "camera_id": Text(),
    "capture_start_local": DateTime(),
    "video_id": Text(),
    "video_name": Text(),
    "activity_type": Text(),
    "n_meva_instances": Integer(),
    "gt_ids": Text(),
    "required_objects": Text(),
    "covering_objects_missing": Text(),
    "n_annotations": Integer(),
    "has_annotation_with_all_objects": Boolean(),
    "n_annotations_with_all_objects": Integer(),
    "coverage_score": Float(),
    "covered": Boolean(),
    "covering_event_id": Text(),
    "covering_event_name": Text(),
    "judge_reason": Text(),
    "status": Text(),
    "judge_backend": Text(),
    "judge_model": Text(),
    "evaluated_at": DateTime(timezone=True),
}

writable = (
    coverage_report[coverage_report["status"] != "judge_missing"][REPORT_COLUMNS]
    .rename(columns=REPORT_RENAMES)
    .copy()
)

# NaN is not a boolean; send a real NULL for pairs that were never scored.
writable["covered"] = writable["covered"].where(writable["covered"].notna(), None)
writable["coverage_score"] = writable["coverage_score"].astype("float64")

if not WRITE_TO_DB:
    print(f"WRITE_TO_DB is False - {len(writable)} rows kept local, nothing written.")
elif writable.empty:
    print("No coverage rows with a verdict - the stored table is left untouched.")
elif VIDEO_FILTER:
    # Replace only the filtered videos' rows: delete and append inside one transaction, so a failed
    # insert rolls the delete back too.
    with engine.begin() as conn:
        deleted = 0
        if report_exists:
            deleted = conn.execute(
                text(f"DELETE FROM bronze.{COVERAGE_TABLE} WHERE video_name LIKE :pattern;"),
                {"pattern": f"%{VIDEO_FILTER}%"},
            ).rowcount
        writable.to_sql(
            COVERAGE_TABLE,
            conn,
            schema="bronze",
            if_exists="append",
            index=False,
            method=psql_insert_copy,
            dtype=REPORT_DTYPES,
        )
    print(f"Replaced rows for {VIDEO_FILTER!r} in bronze.{COVERAGE_TABLE}: "
          f"{deleted} old rows removed, {len(writable)} rows written.")
else:
    # Drop and recreate the table with this run's rows only. pandas runs this in one transaction,
    # so a failed insert leaves the previous table in place.
    writable.to_sql(
        COVERAGE_TABLE,
        engine,
        schema="bronze",
        if_exists="replace",
        index=False,
        method=psql_insert_copy,
        dtype=REPORT_DTYPES,
    )
    print(f"Replaced bronze.{COVERAGE_TABLE}: {existing_rows} old rows removed, {len(writable)} rows written.")

held_back = int((coverage_report["status"] == "judge_missing").sum())
if held_back:
    print(f"Held back {held_back} pairs without a verdict; re-run the notebook to retry them.")

Replaced rows for '2018-03-07.11-00-01.11-05-01.school.G421' in bronze.meva_event_coverage: 14 old rows removed, 14 rows written.


In [25]:
if report_exists or (WRITE_TO_DB and not writable.empty):
    summary = pd.read_sql(
        text(f"""
        WITH latest AS (
            -- one row per pair and judge model: the newest verdict wins
            SELECT DISTINCT ON (pair_id, judge_model) *
            FROM bronze.{COVERAGE_TABLE}
            WHERE status IN ('judged', 'no_annotations')
            ORDER BY pair_id, judge_model, evaluated_at DESC
        )
        SELECT judge_backend,
               judge_model,
               COUNT(*) AS activity_types,
               SUM(CASE WHEN covered THEN 1 ELSE 0 END) AS covered,
               ROUND(AVG(CASE WHEN covered THEN 1.0 ELSE 0.0 END), 3) AS coverage
        FROM latest
        GROUP BY judge_backend, judge_model
        ORDER BY judge_backend;
        """),
        engine,
    )
    print(f"bronze.{COVERAGE_TABLE} contents by judge:")
    display(summary)
else:
    print(f"bronze.{COVERAGE_TABLE} does not exist yet.")

bronze.meva_event_coverage contents by judge:


,judge_backend,judge_model,activity_types,covered,coverage
0,mlx,mlx-community/Qwen3-8B-4bit,605,467,0.772


# Part B — Agreement: do the generated events describe what MEVA recorded at that time?

The opposite direction. Every generated event is matched to the MEVA activities whose frame ranges overlap its
window in the same video, exactly as the earlier matching stage did (`camera_id` + capture time, frame overlap,
activities in chronological order, each turned into a fixed sentence). The judge then compares the annotation
text with those reference sentences using the stepped rubric from the QA notebook: contradiction, wrong actor,
same action, clearly implied, or same actor with the action not described.

An event with no overlapping MEVA activity is recorded as `no_reference` and never sent to the judge; a video
with no ground truth at all gives `has_ground_truth = false`.

## 12. Match generated events to MEVA activities

`VIDEO_FILTER` applies here too: with it set only the matching videos' events are judged and only their rows
for this prompt version are replaced.

The pairs are also written to `bronze.matched_pairs`, the table the QA notebook reads, so it keeps working on
fresh data: all rows are replaced on a full run, only the filtered videos' rows with `VIDEO_FILTER`.


In [ ]:
# One fixed sentence per MEVA activity type, the wording the earlier matching stage used.
MEVA_SENTENCES = {
    "hand_interacts_with_person": "a person's hand interacts with another person",
    "person_carries_heavy_object": "a person carries a heavy object",
    "person_closes_facility_door": "a person closes a facility door",
    "person_closes_trunk": "a person closes a trunk",
    "person_closes_vehicle_door": "a person closes a vehicle door",
    "person_embraces_person": "a person embraces another person",
    "person_enters_scene_through_structure": "a person enters the scene through a structure",
    "person_enters_vehicle": "a person enters a vehicle",
    "person_exits_scene_through_structure": "a person exits the scene through a structure",
    "person_exits_vehicle": "a person exits a vehicle",
    "person_interacts_with_laptop": "a person interacts with a laptop",
    "person_loads_vehicle": "a person loads a vehicle",
    "person_opens_facility_door": "a person opens a facility door",
    "person_opens_trunk": "a person opens a trunk",
    "person_opens_vehicle_door": "a person opens a vehicle door",
    "person_picks_up_object": "a person picks up an object",
    "person_purchases": "a person purchases something",
    "person_puts_down_object": "a person puts down an object",
    "person_reads_document": "a person reads a document",
    "person_rides_bicycle": "a person rides a bicycle",
    "person_sits_down": "a person sits down",
    "person_stands_up": "a person stands up",
    "person_steals_object": "a person steals an object",
    "person_talks_on_phone": "a person talks on a phone",
    "person_talks_to_person": "a person talks to another person",
    "person_texts_on_phone": "a person texts on a phone",
    "person_transfers_object": "a person transfers an object to another person",
    "person_unloads_vehicle": "a person unloads a vehicle",
    "vehicle_drops_off_person": "a vehicle drops off a person",
    "vehicle_makes_u_turn": "a vehicle makes a U-turn",
    "vehicle_picks_up_person": "a vehicle picks up a person",
    "vehicle_reverses": "a vehicle reverses",
    "vehicle_starts": "a vehicle starts moving",
    "vehicle_stops": "a vehicle stops",
    "vehicle_turns_left": "a vehicle turns left",
    "vehicle_turns_right": "a vehicle turns right",
}
IGNORED_MEVA_TYPES = ("empty_37",)   # MEVA placeholder rows, not activities


def meva_sentence(activity_type):
    return MEVA_SENTENCES.get(activity_type, "a " + activity_type.replace("_", " "))


matching_sql = text(f"""
SELECT
    e.event_id,
    e.video_id,
    v.video_name,
    v.camera_id,
    e.event_name    AS model_event_name,
    e.description   AS model_description,
    e.start_frame   AS event_start_frame,
    e.end_frame     AS event_end_frame,
    array_agg(g.activity_type ORDER BY g.start_frame, g.activity_type) FILTER (WHERE g.id IS NOT NULL) AS matched_types,
    EXISTS (SELECT 1 FROM bronze.ground_truth gt WHERE gt.video_id = e.video_id) AS has_ground_truth
FROM bronze.events e
JOIN bronze.videos v ON v.video_id = e.video_id
LEFT JOIN bronze.ground_truth g
    ON g.video_id = e.video_id
   AND NOT (g.activity_type = ANY(:ignored))
   AND g.start_frame <= e.end_frame + :tolerance
   AND g.end_frame   >= e.start_frame - :tolerance
WHERE e.start_frame IS NOT NULL AND e.end_frame IS NOT NULL
  AND (CAST(:pattern AS text) IS NULL OR v.video_name LIKE :pattern)
GROUP BY e.event_id, e.video_id, v.video_name, v.camera_id, e.event_name, e.description, e.start_frame, e.end_frame
ORDER BY v.video_name, e.start_frame, e.event_id;
""")

matched_events = pd.read_sql(
    matching_sql,
    engine,
    params={
        "ignored": list(IGNORED_MEVA_TYPES),
        "tolerance": FRAME_TOLERANCE,
        "pattern": f"%{VIDEO_FILTER}%" if VIDEO_FILTER else None,
    },
)


def unique_in_order(values):
    seen, ordered = set(), []
    for value in values or []:
        if value not in seen:
            seen.add(value)
            ordered.append(value)
    return ordered


matched_events["matched_types"] = matched_events["matched_types"].apply(unique_in_order)
matched_events["reference_list"] = matched_events["matched_types"].apply(lambda types: [meva_sentence(t) for t in types])
matched_events["meva_reference"] = matched_events["reference_list"].apply(
    lambda refs: "; ".join(refs) if refs else "(no matching MEVA ground truth found)"
)
matched_events["n_meva_activities_matched"] = matched_events["matched_types"].apply(len)
# A stable integer id per event, so (id, prompt_version) stays unique the way the QA notebook's table expects.
matched_events["id"] = matched_events["event_id"].apply(lambda event_id: int(str(event_id)[:15], 16))

matched_events["agreement_status"] = "judged"
matched_events.loc[matched_events["n_meva_activities_matched"] == 0, "agreement_status"] = "no_reference"

print("Generated events selected:", len(matched_events), "in", matched_events["video_id"].nunique(), "video(s)")
print(matched_events["agreement_status"].value_counts().to_string())
print("Events in videos with no ground truth at all:", int((~matched_events["has_ground_truth"]).sum()))
matched_events[["video_name", "model_event_name", "event_start_frame", "event_end_frame",
                "n_meva_activities_matched", "meva_reference", "agreement_status"]].head(12)


In [ ]:
MATCHED_PAIRS_COLUMNS = [
    "id", "event_id", "video_id", "camera_id", "model_event_name", "model_description", "event_start_frame",
    "event_end_frame", "meva_reference", "n_meva_activities_matched", "created_at", "has_ground_truth",
]
MATCHED_PAIRS_DDL = f"""
CREATE TABLE IF NOT EXISTS bronze.{MATCHED_PAIRS_TABLE} (
    id BIGINT PRIMARY KEY,
    event_id TEXT,
    camera_id TEXT,
    model_event_name TEXT,
    model_description TEXT,
    event_start_frame INTEGER,
    event_end_frame INTEGER,
    meva_reference TEXT,
    n_meva_activities_matched INTEGER,
    created_at TIMESTAMPTZ DEFAULT now(),
    has_ground_truth BOOLEAN
);
ALTER TABLE bronze.{MATCHED_PAIRS_TABLE} ADD COLUMN IF NOT EXISTS video_id TEXT;
"""

matched_pairs_rows = matched_events[MATCHED_PAIRS_COLUMNS[:2] + MATCHED_PAIRS_COLUMNS[3:-2] + ["has_ground_truth"]].copy()
matched_pairs_rows["video_id"] = matched_events["video_id"]
matched_pairs_rows["created_at"] = datetime.now(timezone.utc)
matched_pairs_rows = matched_pairs_rows[MATCHED_PAIRS_COLUMNS]

if not WRITE_TO_DB:
    print(f"WRITE_TO_DB is False - {len(matched_pairs_rows)} matched pairs kept local, nothing written.")
elif matched_pairs_rows.empty:
    print("No matched pairs to write.")
else:
    # Same rule as every other table: replace the filtered videos' rows, or everything, in one transaction.
    with engine.begin() as conn:
        for statement in MATCHED_PAIRS_DDL.strip().split(";"):
            if statement.strip():
                conn.execute(text(statement))
        if VIDEO_FILTER:
            deleted = conn.execute(
                text(f"DELETE FROM bronze.{MATCHED_PAIRS_TABLE} WHERE event_id = ANY(:ids) OR video_id = ANY(:videos);"),
                {"ids": list(matched_events["event_id"]), "videos": list(matched_events["video_id"].unique())},
            ).rowcount
        else:
            deleted = conn.execute(text(f"DELETE FROM bronze.{MATCHED_PAIRS_TABLE};")).rowcount
        matched_pairs_rows.to_sql(
            MATCHED_PAIRS_TABLE, conn, schema="bronze", if_exists="append", index=False, method=psql_insert_copy,
        )
    print(f"bronze.{MATCHED_PAIRS_TABLE}: {deleted} old rows removed, {len(matched_pairs_rows)} rows written.")


## 13. The agreement judge

The rubric is the QA notebook's `v2.3_semantic_judge`, unchanged: it accounts for the model describing *where things
move* while MEVA names *what happens*, so an accurate movement-only description scores 0.50 rather than 0. The
hosted judge takes `AGREEMENT_BATCH_SIZE` events per call with a strict JSON schema; the local judge takes one
event per call. `llm_match` is computed in Python as `score >= AGREEMENT_THRESHOLD`.


In [ ]:
AGREEMENT_RULES = """You are evaluating generated CCTV event annotations against MEVA reference annotations.

The generated annotation (name + description) usually describes appearance, position and movement.
The MEVA reference lists one or more fixed activities joined by ";". Compare the annotation with EACH
reference activity and give the score of the BEST-matching one.

Apply these steps in order and stop at the first that applies:

STEP 1 -> 0.00 (contradiction)
  The description explicitly states the OPPOSITE of the same kind of action in the reference:
  exits vs enters (a vehicle or the scene), opens vs closes, picks up vs puts down, loads vs unloads,
  turns left vs turns right.
  NOT a contradiction: an actor being idle, standing, parked or simply present. Other people or vehicles
  in the scene may be doing the reference activity.

STEP 2 -> 0.25 (different actor)
  No reference activity has the same kind of actor as the description.
  Actor = the subject of the activity: "a vehicle ..." is a vehicle activity, "a person ..." is a person
  activity (including person-vehicle interactions like "a person opens a vehicle door").
  If the description covers both people and vehicles, both actor types count.

STEP 3 -> 1.00 (same action)
  The description states a reference action, in the same or equivalent words.

STEP 4 -> 0.75 (clearly implied)
  The description gives SPECIFIC evidence of a reference action in different words, for example:
    "curves around and heads back the way it came" -> makes a U-turn
    "pulls into a space and comes to rest" -> stops
    "walks through the doorway into the building" -> exits the scene through a structure
  The evidence must be in the text. Plausible or possible is NOT enough.

STEP 5 -> 0.50 (same actor, action not described)
  Same actor type, no contradiction, but the reference action is not described or clearly implied.
  Descriptions of movement, position, appearance, standing or being present belong here.

Activity rules (a description only counts for these if it describes the change itself):
- "starts moving": a stationary vehicle begins to move. Moving across/through the frame is NOT starting.
- "stops": a moving vehicle comes to a halt. Parked or stationary the whole time is NOT stopping.
- turns / U-turn: a change of direction. Moving from one side of the frame to another is NOT a turn.
- "reverses": the vehicle backs up.
- "enters the scene through a structure": a person comes OUT of a door/building INTO view.
- "exits the scene through a structure": a person goes INTO a door/building and leaves view.
  Approaching or standing near an entrance is neither.
- carries / picks up / puts down / loads / unloads: the object must be mentioned.
- "appears", "is visible", "is detected", "enters the frame" are NOT activities.

Evaluate each annotation independently. Return one result per id.
reason: at most 15 words; name the reference activity compared and the step applied,
e.g. "vehicle stops: same actor, moving only, no stop described (step 5)"."""

AGREEMENT_SCORE_LABELS = ["1.00", "0.75", "0.50", "0.25", "0.00"]
AGREEMENT_RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer"},
                    "score": {"type": "string", "enum": AGREEMENT_SCORE_LABELS},
                    "reason": {"type": "string"},
                },
                "required": ["id", "score", "reason"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["results"],
    "additionalProperties": False,
}
AGREEMENT_MAX_RETRIES = 4
AGREEMENT_MAX_WAIT_SECONDS = 120
AGREEMENT_SLEEP_BETWEEN_CALLS = 2.0


def annotation_payload(row):
    return {
        "id": int(row["id"]),
        "generated_name": row["model_event_name"],
        "generated_description": row["model_description"],
        "meva_reference": "; ".join(row["reference_list"]),
    }


def snap_agreement_score(value):
    """Nearest allowed score, or None when the value is unusable."""
    try:
        return min(ALLOWED_SCORES, key=lambda allowed: abs(allowed - float(value)))
    except (TypeError, ValueError):
        return None


def agreement_groq_batch(batch_df):
    """One hosted call for a batch of events, strict JSON output; retried on rate limits and timeouts."""
    import time

    import groq

    retryable = tuple(
        error for error in (
            getattr(groq, "RateLimitError", None), getattr(groq, "APITimeoutError", None),
            getattr(groq, "APIConnectionError", None), getattr(groq, "InternalServerError", None),
        ) if error is not None
    )
    payload = {"annotations": [annotation_payload(row) for _, row in batch_df.iterrows()]}
    for attempt in range(1, AGREEMENT_MAX_RETRIES + 1):
        try:
            response = groq_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[
                    {"role": "system", "content": AGREEMENT_RULES},
                    {"role": "user", "content": json.dumps(payload, separators=(",", ":"))},
                ],
                temperature=0,
                reasoning_effort="low",
                max_completion_tokens=4096,
                response_format={
                    "type": "json_schema",
                    "json_schema": {"name": "annotation_judgements", "strict": True, "schema": AGREEMENT_RESPONSE_SCHEMA},
                },
            )
            break
        except retryable as error:
            headers = getattr(getattr(error, "response", None), "headers", None) or {}
            try:
                wait = float(headers.get("retry-after")) if headers.get("retry-after") else min(60, 2 ** attempt)
            except ValueError:
                wait = min(60, 2 ** attempt)
            if wait > AGREEMENT_MAX_WAIT_SECONDS:
                raise RuntimeError(f"Groq asked to wait {wait:.0f}s (likely a daily limit); rerun later.") from error
            if attempt == AGREEMENT_MAX_RETRIES:
                raise
            print(f"    {type(error).__name__}; retry {attempt}/{AGREEMENT_MAX_RETRIES - 1} in {wait:.0f}s", flush=True)
            time.sleep(wait)
    choice = response.choices[0]
    if choice.finish_reason == "length":
        raise ValueError("Response truncated; lower AGREEMENT_BATCH_SIZE.")
    results = json.loads(choice.message.content).get("results", [])
    expected = {int(row["id"]) for _, row in batch_df.iterrows()}
    verdicts = {}
    for item in results:
        try:
            item_id = int(item.get("id"))
        except (TypeError, ValueError):
            continue
        score = snap_agreement_score(item.get("score"))
        if item_id in expected and item_id not in verdicts and score is not None:
            verdicts[item_id] = {"score": score, "reason": str(item.get("reason") or "").strip()}
    return verdicts


def agreement_mlx_event(row):
    """One local call per event: the same rubric, one annotation, a JSON object back."""
    prompt = (
        AGREEMENT_RULES
        + "\n\nANNOTATION TO EVALUATE:\n"
        + json.dumps(annotation_payload(row), indent=2)
        + "\n\nAnswer with one JSON object and nothing else:\n"
        + '  "score": one of 1.0, 0.75, 0.5, 0.25, 0.0\n'
        + '  "reason": one short sentence naming the reference activity compared and the step applied\n\n'
        + "JSON object:"
    )
    output = mlx_generate(
        mlx_model, mlx_tokenizer,
        prompt=mlx_tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}], add_generation_prompt=True, enable_thinking=False,
        ),
        max_tokens=MLX_MAX_TOKENS, sampler=mlx_sampler, verbose=False,
    )
    verdict = parse_json_object(output)
    score = snap_agreement_score(verdict.get("score"))
    if score is None:
        raise ValueError(f"Unusable verdict: {verdict}")
    return {int(row["id"]): {"score": score, "reason": str(verdict.get("reason") or "").strip()}}


def judge_agreement(events_df):
    """Judge every event with the selected backend; a failed batch is split in halves down to single events."""
    import time

    verdicts, failures = {}, {}

    def run_batch(batch):
        try:
            if JUDGE_BACKEND == "groq":
                got = agreement_groq_batch(batch)
            else:
                got = {}
                for _, row in batch.iterrows():
                    got.update(agreement_mlx_event(row))
        except RuntimeError:
            raise
        except Exception as error:
            got, problem = {}, f"{type(error).__name__}: {error}"
        else:
            problem = "no usable verdict returned"
        verdicts.update(got)
        missing = batch[~batch["id"].astype(int).isin(got)]
        if missing.empty:
            return
        if len(batch) == 1:
            failures[int(missing.iloc[0]["id"])] = problem
            return
        middle = max(1, len(missing) // 2)
        for part in (missing.iloc[:middle], missing.iloc[middle:]):
            if not part.empty:
                run_batch(part)

    size = AGREEMENT_BATCH_SIZE if JUDGE_BACKEND == "groq" else 1
    batches = [events_df.iloc[start:start + size] for start in range(0, len(events_df), size)]
    for position, batch in enumerate(batches, start=1):
        run_batch(batch)
        if JUDGE_BACKEND == "groq" or position % 20 == 0 or position == len(batches):
            print(f"Judged batch {position}/{len(batches)} ({len(verdicts)} verdicts so far)", flush=True)
        if JUDGE_BACKEND == "groq" and position < len(batches):
            time.sleep(AGREEMENT_SLEEP_BETWEEN_CALLS)
    return verdicts, failures


to_judge = matched_events[matched_events["agreement_status"] == "judged"]
agreement_verdicts, agreement_failures = judge_agreement(to_judge)

print(f"\nEvents sent to the judge: {len(to_judge)}")
print(f"Verdicts returned:        {len(agreement_verdicts)}")
print(f"Not sent (no reference):  {int((matched_events['agreement_status'] == 'no_reference').sum())}")
if agreement_failures:
    print(f"Failed after retries:     {len(agreement_failures)} (left out of the table; rerun to retry)")


## 14. Build the agreement report

One row per generated event, in the shape of `bronze.annotations_report` so the QA notebook's analysis still
works: the matched-pair columns plus `llm_score`, `llm_match`, `llm_reason`, `eval_status`, `prompt_version`,
`model`, `evaluated_at`. Events the judge failed on are `judge_missing` and are not written.


In [ ]:
agreement_report = matched_events.copy()
agreement_report["llm_score"] = agreement_report["id"].map(lambda i: agreement_verdicts.get(int(i), {}).get("score"))
agreement_report["llm_reason"] = agreement_report["id"].map(lambda i: agreement_verdicts.get(int(i), {}).get("reason"))
agreement_report["llm_match"] = agreement_report["llm_score"].map(
    lambda score: None if pd.isna(score) else bool(score >= AGREEMENT_THRESHOLD)
).astype("object")

agreement_report["eval_status"] = "scored"
no_reference = agreement_report["agreement_status"] == "no_reference"
agreement_report.loc[no_reference, "eval_status"] = "no_reference"
agreement_report.loc[no_reference, "llm_reason"] = "No MEVA reference to compare against."
missing_verdict = (agreement_report["eval_status"] == "scored") & agreement_report["llm_score"].isna()
agreement_report.loc[missing_verdict, "eval_status"] = "judge_missing"

agreement_report["prompt_version"] = AGREEMENT_PROMPT_VERSION
agreement_report["model"] = JUDGE_MODEL
agreement_report.loc[no_reference, "model"] = None
agreement_report["created_at"] = datetime.now(timezone.utc)
agreement_report["evaluated_at"] = datetime.now(timezone.utc)

print("Agreement rows:", len(agreement_report))
print(agreement_report["eval_status"].value_counts().to_string())
agreement_report[["video_name", "model_event_name", "meva_reference", "llm_score", "llm_match", "llm_reason", "eval_status"]].head(30)


## 15. Agreement metrics

Match rate and score distribution over judged events, then per video. A pile of 0.50 scores means the descriptions
are consistent with MEVA but describe movement rather than the MEVA action.


In [ ]:
scored = agreement_report[agreement_report["eval_status"] == "scored"].copy()
scored["llm_match"] = scored["llm_match"].astype(bool)

print(f"Generated events in scope:        {len(agreement_report)}")
print(f"  judged against a MEVA reference: {len(scored)}")
print(f"  no MEVA reference in window:     {int((agreement_report['eval_status'] == 'no_reference').sum())}")
print(f"  judge failed:                    {int((agreement_report['eval_status'] == 'judge_missing').sum())}")
if len(scored):
    print(f"\nMean score (judged):              {scored['llm_score'].mean():.3f}")
    print(f"Agreement rate (score >= {AGREEMENT_THRESHOLD}): {scored['llm_match'].mean():.1%}")
    print("\nScore distribution:")
    print(scored["llm_score"].value_counts().reindex([1.0, 0.75, 0.5, 0.25, 0.0], fill_value=0).to_string())

agreement_by_video = (
    scored.groupby(["camera_id", "video_name"])
    .agg(events=("id", "count"), matched=("llm_match", "sum"), mean_score=("llm_score", "mean"))
    .assign(agreement=lambda df: (df["matched"] / df["events"]).round(3), mean_score=lambda df: df["mean_score"].round(3))
    .sort_values("agreement")
)
print("\nAgreement by video (worst first):")
agreement_by_video


In [ ]:
# Events the judge scored lowest: what the pipeline described versus what MEVA says happened then.
scored.sort_values("llm_score")[
    ["video_name", "event_start_frame", "event_end_frame", "model_event_name", "model_description",
     "meva_reference", "llm_score", "llm_reason"]
].head(25)


## 16. Write to `bronze.annotations_report`

The table keeps every prompt version, so earlier judges stay comparable. This run replaces the rows of
`AGREEMENT_PROMPT_VERSION` for the events in scope (all events, or the filtered videos' events) in one
transaction, and leaves other prompt versions untouched. `judge_missing` rows are not written. The table is
created with the QA notebook's columns when it does not exist yet.


In [ ]:
AGREEMENT_COLUMNS = [
    "id", "event_id", "camera_id", "model_event_name", "model_description", "event_start_frame",
    "event_end_frame", "meva_reference", "n_meva_activities_matched", "created_at", "has_ground_truth",
    "llm_score", "llm_match", "llm_reason", "eval_status", "prompt_version", "model", "evaluated_at",
]
AGREEMENT_DDL = f"""
CREATE TABLE IF NOT EXISTS bronze.{AGREEMENT_TABLE} (
    id BIGINT,
    event_id TEXT,
    camera_id TEXT,
    model_event_name TEXT,
    model_description TEXT,
    event_start_frame INTEGER,
    event_end_frame INTEGER,
    meva_reference TEXT,
    n_meva_activities_matched INTEGER,
    created_at TIMESTAMPTZ,
    has_ground_truth BOOLEAN
);
ALTER TABLE bronze.{AGREEMENT_TABLE} ADD COLUMN IF NOT EXISTS llm_score DOUBLE PRECISION;
ALTER TABLE bronze.{AGREEMENT_TABLE} ADD COLUMN IF NOT EXISTS llm_match BOOLEAN;
ALTER TABLE bronze.{AGREEMENT_TABLE} ADD COLUMN IF NOT EXISTS llm_reason TEXT;
ALTER TABLE bronze.{AGREEMENT_TABLE} ADD COLUMN IF NOT EXISTS eval_status TEXT;
ALTER TABLE bronze.{AGREEMENT_TABLE} ADD COLUMN IF NOT EXISTS prompt_version TEXT;
ALTER TABLE bronze.{AGREEMENT_TABLE} ADD COLUMN IF NOT EXISTS model TEXT;
ALTER TABLE bronze.{AGREEMENT_TABLE} ADD COLUMN IF NOT EXISTS evaluated_at TIMESTAMPTZ;
CREATE UNIQUE INDEX IF NOT EXISTS {AGREEMENT_TABLE}_id_prompt_version_uq ON bronze.{AGREEMENT_TABLE} (id, prompt_version);
"""

agreement_writable = agreement_report[agreement_report["eval_status"] != "judge_missing"][AGREEMENT_COLUMNS].copy()
agreement_writable["llm_match"] = agreement_writable["llm_match"].where(agreement_writable["llm_match"].notna(), None)
agreement_writable["llm_score"] = agreement_writable["llm_score"].astype("float64")

if not WRITE_TO_DB:
    print(f"WRITE_TO_DB is False - {len(agreement_writable)} rows kept local, nothing written.")
elif agreement_writable.empty:
    print("No agreement rows with a verdict - the stored table is left untouched.")
else:
    with engine.begin() as conn:
        for statement in AGREEMENT_DDL.strip().split(";"):
            if statement.strip():
                conn.execute(text(statement))
        if VIDEO_FILTER:
            deleted = conn.execute(
                text(f"DELETE FROM bronze.{AGREEMENT_TABLE} WHERE prompt_version = :pv AND event_id = ANY(:ids);"),
                {"pv": AGREEMENT_PROMPT_VERSION, "ids": list(matched_events["event_id"])},
            ).rowcount
        else:
            deleted = conn.execute(
                text(f"DELETE FROM bronze.{AGREEMENT_TABLE} WHERE prompt_version = :pv;"),
                {"pv": AGREEMENT_PROMPT_VERSION},
            ).rowcount
        agreement_writable.to_sql(
            AGREEMENT_TABLE, conn, schema="bronze", if_exists="append", index=False, method=psql_insert_copy,
        )
    print(f"bronze.{AGREEMENT_TABLE} ({AGREEMENT_PROMPT_VERSION}): {deleted} old rows removed, {len(agreement_writable)} rows written.")

held_back = int((agreement_report["eval_status"] == "judge_missing").sum())
if held_back:
    print(f"Held back {held_back} event(s) without a verdict; re-run the notebook to retry them.")


In [ ]:
if WRITE_TO_DB:
    print(f"bronze.{AGREEMENT_TABLE} contents by prompt version:")
    display(pd.read_sql(
        text(f"""
        SELECT prompt_version, eval_status, COUNT(*) AS rows,
               ROUND(AVG(llm_score)::numeric, 3) AS avg_llm_score,
               ROUND(AVG(CASE WHEN llm_match THEN 1.0 ELSE 0.0 END)::numeric, 3) AS match_rate
        FROM bronze.{AGREEMENT_TABLE}
        GROUP BY prompt_version, eval_status
        ORDER BY prompt_version, eval_status;
        """),
        engine,
    ))


# Both directions together

## 17. Coverage and agreement side by side

Coverage (Part A) is the share of MEVA activity types the pipeline described at least once per video. Agreement
(Part B) is the share of generated events whose text matches a MEVA activity in their own time window. Together
they read like recall and precision: a pipeline can score well on one by sacrificing the other, so report both.


In [ ]:
coverage_pairs = len(evaluable)
coverage_rate = evaluable["covered"].mean() if coverage_pairs else float("nan")
agreement_events = len(scored)
agreement_rate = scored["llm_match"].mean() if agreement_events else float("nan")

summary_both = pd.DataFrame([
    {"measure": f"Coverage (MEVA activity types described, score >= {COVERAGE_THRESHOLD})", "n": coverage_pairs,
     "rate": round(float(coverage_rate), 3) if coverage_pairs else None, "table": f"bronze.{COVERAGE_TABLE}"},
    {"measure": f"Agreement (generated events matching MEVA, score >= {AGREEMENT_THRESHOLD})", "n": agreement_events,
     "rate": round(float(agreement_rate), 3) if agreement_events else None, "table": f"bronze.{AGREEMENT_TABLE}"},
])
print(f"Judge: {JUDGE_MODEL} ({JUDGE_BACKEND})" + (f" · videos matching {VIDEO_FILTER!r}" if VIDEO_FILTER else " · all videos"))
display(summary_both)

per_video_coverage = (
    evaluable.groupby("video_name")
    .agg(activity_types=("pair_id", "count"), covered=("covered", "sum"))
    .assign(coverage=lambda df: (df["covered"] / df["activity_types"]).round(3))
)
per_video_agreement = (
    scored.groupby("video_name")
    .agg(events_judged=("id", "count"), matched=("llm_match", "sum"))
    .assign(agreement=lambda df: (df["matched"] / df["events_judged"]).round(3))
)
side_by_side = per_video_coverage.join(per_video_agreement, how="outer").sort_values("coverage")
print("\nPer video (worst coverage first):")
side_by_side
